In [0]:
# =============================================================
# Notebook : 02_inventory_silver.py
# Purpose  : Bronze → Silver for daily inventory snapshots
# Source   : bronze/batch/inventory/
# Target   : silver/fact_inventory_daily/ (Delta)
# =============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

BRONZE_PATH = "abfss://bronze@walmartdata.dfs.core.windows.net/batch/inventory/*/"
SILVER_PATH = "abfss://silver@walmartdata.dfs.core.windows.net/fact_inventory_daily/"

# ── Read all partitions ───────────────────────────────────────
df_raw = spark.read.parquet(BRONZE_PATH)
print(f"Raw rows: {df_raw.count():,}")
display(df_raw.limit(3))

In [0]:
# ── Schema enforcement + derived columns ─────────────────────
df_clean = (
    df_raw
    .select(
        F.to_date("snapshot_date").alias("snapshot_date"),
        F.col("store_id").cast(StringType()),
        F.col("store_city").cast(StringType()),
        F.col("store_region").cast(StringType()),
        F.col("store_size").cast(StringType()),
        F.col("sku").cast(StringType()),
        F.col("product_name").cast(StringType()),
        F.col("category").cast(StringType()),
        F.col("supplier_id").cast(StringType()),
        F.col("opening_stock").cast(IntegerType()),
        F.col("units_sold").cast(IntegerType()),
        F.col("units_received").cast(IntegerType()),
        F.col("closing_stock").cast(IntegerType()),
        F.col("reorder_point").cast(IntegerType()),
        F.col("reorder_qty").cast(IntegerType()),
        F.col("cost_price").cast(DoubleType()),
        F.col("selling_price").cast(DoubleType()),
        F.col("gross_margin_pct").cast(DoubleType()),
        F.col("stock_status").cast(StringType()),
        F.col("warehouse_id").cast(StringType()),
        F.col("source_system").cast(StringType()),
    )
    # ── Derived business columns ──────────────────────────────
    .withColumn("days_of_stock_remaining",
                F.when(F.col("units_sold") > 0,
                       F.col("closing_stock") / F.col("units_sold"))
                 .otherwise(None))
    .withColumn("needs_reorder",
                F.col("closing_stock") < F.col("reorder_point"))
    .withColumn("sellthrough_rate",
                F.when(F.col("opening_stock") > 0,
                       F.round(F.col("units_sold") /
                               F.col("opening_stock") * 100, 2))
                 .otherwise(0.0))
    .withColumn("inventory_value",
                F.round(F.col("closing_stock") *
                        F.col("cost_price"), 2))
    .withColumn("stockout_risk",
                F.when(F.col("closing_stock") == 0, "STOCKOUT")
                 .when(F.col("closing_stock") < F.col("reorder_point") * 0.5, "CRITICAL")
                 .when(F.col("closing_stock") < F.col("reorder_point"), "LOW")
                 .otherwise("ADEQUATE"))
    # ── Audit columns ─────────────────────────────────────────
    .withColumn("silver_processed_at", F.current_timestamp())
    .withColumn("pipeline_version", F.lit("inventory_bronze_to_silver_v1"))
    # ── Remove bad rows ───────────────────────────────────────
    .filter(F.col("snapshot_date").isNotNull())
    .filter(F.col("store_id").isNotNull())
    .filter(F.col("sku").isNotNull())
    .filter(F.col("closing_stock") >= 0)
)

print(f"Clean rows : {df_clean.count():,}")
print(f"\nStockout risk summary:")
df_clean.groupBy("stockout_risk").count().orderBy("count", ascending=False).display()

In [0]:
# ── Write Silver Delta + register in Unity Catalog ────────────
(
    df_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("snapshot_date", "store_region")
    .save(SILVER_PATH)
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS walmart_silver.fact_inventory_daily
    USING DELTA
    LOCATION '{SILVER_PATH}'
""")

print(f"✅ walmart_silver.fact_inventory_daily registered")
print(f"   Rows: {spark.table('walmart_silver.fact_inventory_daily').count():,}")

# Show stores needing reorder right now
print("\n⚠️  Items needing reorder:")
spark.sql("""
    SELECT store_id, sku, product_name,
           closing_stock, reorder_point, stockout_risk
    FROM walmart_silver.fact_inventory_daily
    WHERE needs_reorder = true
    ORDER BY stockout_risk, closing_stock
    LIMIT 10
""").display()